# 08 — Time series III: Box-Jenkins, VAR, GARCH (1970–1986)

## Usage and numerical contracts
ARIMA uses Hannan-Rissanen, not a general exact likelihood. VAR requires independent regressors for the QR; the available stability check is not a general spectral solver. GARCH variance targeting is a grid search: candidates in T, finite shocks and a positive finite mean of squares; no guarantee of a global maximum. Simulation by default generates Box-Muller in double and converts to T; the innovationSource overloads accept noise directly in T, with distribution and standardisation left to the caller. Do not use the shared noise as proof of all the Decimal128 digits.
[API catalogue](12-api-reference.ipynb), [checks on the six formats](11-consolidation.ipynb), [limits and coverage](../../ARCHITECTURE.md).
**JupyterSharp** kernel (C#) on the **.NET 11** runtime. Prerequisite: `dotnet build src/MathemaSharp.slnx -c Release`. From a terminal: `scripts/run-notebooks.ps1` (sets the roll-forward to runtime 11); in VS Code you need `DOTNET_ROLL_FORWARD=LatestMajor` and, as long as .NET 11 is RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

**Box-Jenkins (1970)**: systematic identification, estimation and residual checking. **VAR (Sims, 1980)**: several series, each explained by its own lags and those of the others. **ARCH/GARCH (Engle 1982, Bollerslev 1986)**: the conditional *variance* is modelled, not the mean.

In [ ]:
#load "../shared/MathemaSharp.PlotSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using MathemaSharp.TimeSeries;
// ARIMA(1,1,1): the differences follow an ARMA(1,1) with φ = 0.5, θ = 0.4; the series is their cumulative sum.
var rng = new Random(1970);
var w = Simulation.Arma<double>(rng, 400, 0.2, new[] { 0.5 }, new[] { 0.4 }, 1);
var y = new double[w.Length + 1];
y[0] = 100;
for (var t = 0; t < w.Length; t++) y[t + 1] = y[t] + w[t];
var dy = SeriesStatistics.Difference(y);
var band = 1.96 / Math.Sqrt(dy.Length);
Chart.Lines("Integrated series: non-stationary in mean", new Line("y", y));
Chart.Stems("ACF of y: very slow decay ⇒ difference", SeriesStatistics.Acf(y, 30)[1..], 1.96 / Math.Sqrt(y.Length));
Chart.Stems("ACF of Δy", SeriesStatistics.Acf(dy, 30)[1..], band);
Chart.Stems("PACF of Δy", SeriesStatistics.Pacf(dy, 30), band);

Integrated series: non-stationary in mean 50 100 150 200 250 300 0 100 200 300 400 <path d='M64 177.6L65.58 178.33L67.16 179.9L68.74 181.16L70.32 182.98L71.9 183.41L73.48 183.71L75.06 183.7L76.64 184.47L78.22 185.12L79.8 186.36L81.38 187.72L82.96 188.5L84.54 189.39L86.12 190.15L87.7 189.48L89.28 187.04L90.86 185.4L92.44 185.31L94.02 185.71L95.6 187.49L97.18 188.4L98.76 188.74L100.34 188.99L101.92 189.6L103.5 190.59L105.08 192.31L106.66 194.3L108.24 194.14L109.82 193.2L111.4 192.64L112.98 192.16L114.56 190.88L116.14 190.03L117.72 188.99L119.3 188.81L120.88 189.33L122.46 189.47L124.04 189.66L125.62 190.28L127.2 191.36L128.78 192.1L130.36 190.98L131.94 190.05L133.52 190.22L135.1 190.82L136.68 190.95L138.26 190.9L139.84 189.13L141.42 187.45L143 186.71L144.58 186.3L146.16 186.94L147.74 188.03L149.32 188.84L150.9 189.63L152.48 189.56L154.06 188.31L155.64 187.78L157.22 187.29L158.8 187.54L160.38 186.42L161.96 184.09L163.54 182.6L165.12 182.39L166.7 182.16L168.28 181.93L169.86 181.28L171.44 180L173.02 179.68L174.6 179.73L176.18 179L177.76 178.09L179.34 177.22L180.92 176.12L182.5 175.71L184.08 176.13L185.66 177.15L187.24 176.91L188.82 175.91L190.4 174.95L191.98 175.84L193.56 176.46L195.14 176L196.72 175.08L198.3 173.46L199.88 171.79L201.46 170.17L203.04 168.69L204.62 166.48L206.2 164.17L207.78 162.91L209.36 163.53L210.94 163.9L212.52 163.07L214.1 162.6L215.68 162.92L217.26 163.85L218.84 163.6L220.42 162.3L222 160.81L223.58 160.54L225.16 160.47L226.74 159.06L228.32 158.02L229.9 157.27L231.48 156.92L233.06 156.45L234.64 156.12L236.22 156.64L237.8 157.96L239.38 159.34L240.96 159.61L242.54 159.26L244.12 158.79L245.7 158.56L247.28 159.21L248.86 160.42L250.44 161.3L252.02 161.47L253.6 162.27L255.18 163.04L256.76 163.78L258.34 163.55L259.92 163.84L261.5 164.15L263.08 164.33L264.66 163.88L266.24 162.53L267.82 161.01L269.4 160.39L270.98 160.46L272.56 159.98L274.14 160.37L275.72 160.97L277.3 161.46L278.88 162.35L280.46 162.73L282.04 161.59L283.62 160.59L285.2 161.06L286.78 161.01L288.36 160.07L289.94 159.37L291.52 159.1L293.1 159.25L294.68 159.25L296.26 158.15L297.84 158.2L299.42 160.11L301 160.77L302.58 159.15L304.16 157.5L305.74 157.16L307.32 157.25L308.9 157.19L310.48 157.16L312.06 156.52L313.64 155.84L315.22 156.38L316.8 155.9L318.38 155.38L319.96 154.68L321.54 154.48L323.12 153.78L324.7 153.52L326.28 153.62L327.86 153.7L329.44 154.05L331.02 153.81L332.6 153.79L334.18 153.4L335.76 151.99L337.34 151.1L338.92 150.28L340.5 149.29L342.08 149.05L343.66 149.54L345.24 150.82L346.82 151.41L348.4 150.86L349.98 151.32L351.56 152.57L353.14 154.1L354.72 155.32L356.3 156L357.88 155.84L359.46 155.83L361.04 156.54L362.62 157.22L364.2 158.39L365.78 158.82L367.36 159.14L368.94 160.79L370.52 162.43L372.1 161.83L373.68 160.19L375.26 158.4L376.84 157.31L378.42 156.64L380 155.76L381.58 155.04L383.16 154.07L384.74 153.47L386.32 152.42L387.9 152.21L389.48 153.53L391.06 154.07L392.64 153.96L394.22 154.9L395.8 155.43L397.38 155.17L398.96 155.8L400.54 155.08L402.12 153.58L403.7 151.68L405.28 150.34L406.86 148.99L408.44 148.64L410.02 148.18L411.6 147.73L413.18 146.92L414.76 147.62L416.34 148.32L417.92 148.38L419.5 148.2L421.08 149.15L422.66 149.63L424.24 148.74L425.82 147.75L427.4 146.53L428.98 145.82L430.56 145.61L432.14 145.28L433.72 144.69L435.3 144.83L436.88 144.52L438.46 144.67L440.04 144.41L441.62 143.94L443.2 142.97L444.78 142.48L446.36 141.95L447.94 141L449.52 140.24L451.1 140.2L452.68 140.89L454.26 140.11L455.84 139.25L457.42 139.09L459 138.56L460.58 137.82L462.16 137.85L463.74 137.51L465.32 136.99L466.9 137.57L468.48 138.02L470.06 137.33L471.64 136.07L473.22 134.66L474.8 133.57L476.38 131.99L477.96 130.04L479.54 128.61L481.12 128.11L482.7 128.45L484.28 129.15L485.86 130.79L487.44 132.33L489.02 133.17L490.6 134.22L492.18 135.41L493.76 135.91L495.34 136.95L496.92 136.44L498.5 135L500.08 133.46L501.66 132.09L503.24 131.27L504.82 131.92L506.4 132.51L507.98 133.57L509.56 134.67L511.14 134.15L512.72 132.11L514.3 130.92L515.88 130.75L517.46 131.11L519.04 130.75L520

ACF of y: very slow decay ⇒ difference -0.25 0 0.25 0.5 0.75 1 10 20 30 Lag ACF of y: very slow decay ⇒ difference · x=1: 0.991 ACF of y: very slow decay ⇒ difference · x=2: 0.982 ACF of y: very slow decay ⇒ difference · x=3: 0.973 ACF of y: very slow decay ⇒ difference · x=4: 0.963 ACF of y: very slow decay ⇒ difference · x=5: 0.953 ACF of y: very slow decay ⇒ difference · x=6: 0.943 ACF of y: very slow decay ⇒ difference · x=7: 0.933 ACF of y: very slow decay ⇒ difference · x=8: 0.924 ACF of y: very slow decay ⇒ difference · x=9: 0.914 ACF of y: very slow decay ⇒ difference · x=10: 0.904 ACF of y: very slow decay ⇒ difference · x=11: 0.895 ACF of y: very slow decay ⇒ difference · x=12: 0.885 ACF of y: very slow decay ⇒ difference · x=13: 0.876 ACF of y: very slow decay ⇒ difference · x=14: 0.866 ACF of y: very slow decay ⇒ difference · x=15: 0.857 ACF of y: very slow decay ⇒ difference · x=16: 0.847 ACF of y: very slow decay ⇒ difference · x=17: 0.837 ACF of y: very slow decay ⇒ difference · x=18: 0.827 ACF of y: very slow decay ⇒ difference · x=19: 0.817 ACF of y: very slow decay ⇒ difference · x=20: 0.807 ACF of y: very slow decay ⇒ difference · x=21: 0.796 ACF of y: very slow decay ⇒ difference · x=22: 0.786 ACF of y: very slow decay ⇒ difference · x=23: 0.776 ACF of y: very slow decay ⇒ difference · x=24: 0.765 ACF of y: very slow decay ⇒ difference · x=25: 0.755 ACF of y: very slow decay ⇒ difference · x=26: 0.745 ACF of y: very slow decay ⇒ difference · x=27: 0.734 ACF of y: very slow decay ⇒ difference · x=28: 0.723 ACF of y: very slow decay ⇒ difference · x=29: 0.712 ACF of y: very slow decay ⇒ difference · x=30: 0.701 +band · x=0.5: 0.098 +band · x=30.5: 0.098 -band · x=0.5: -0.098 -band · x=30.5: -0.098 -band +band ACF of y: very slow decay ⇒ difference

ACF of Δy -0.2 0 0.2 0.4 0.6 0.8 10 20 30 Lag ACF of Δy · x=1: 0.666 ACF of Δy · x=2: 0.267 ACF of Δy · x=3: 0.118 ACF of Δy · x=4: 0.087 ACF of Δy · x=5: 0.034 ACF of Δy · x=6: -0.039 ACF of Δy · x=7: -0.101 ACF of Δy · x=8: -0.048 ACF of Δy · x=9: 0.069 ACF of Δy · x=10: 0.099 ACF of Δy · x=11: 0.09 ACF of Δy · x=12: 0.103 ACF of Δy · x=13: 0.146 ACF of Δy · x=14: 0.157 ACF of Δy · x=15: 0.138 ACF of Δy · x=16: 0.11 ACF of Δy · x=17: 0.089 ACF of Δy · x=18: 0.076 ACF of Δy · x=19: 0.076 ACF of Δy · x=20: 0.039 ACF of Δy · x=21: 0.002 ACF of Δy · x=22: 0.025 ACF of Δy · x=23: 0.052 ACF of Δy · x=24: 0.037 ACF of Δy · x=25: 0.04 ACF of Δy · x=26: 0.032 ACF of Δy · x=27: 0.044 ACF of Δy · x=28: 0.061 ACF of Δy · x=29: 0.048 ACF of Δy · x=30: 0.02 +band · x=0.5: 0.098 +band · x=30.5: 0.098 -band · x=0.5: -0.098 -band · x=30.5: -0.098 -band +band ACF of Δy

PACF of Δy -0.4 -0.2 0 0.2 0.4 0.6 0.8 10 20 30 Lag PACF of Δy · x=1: 0.666 PACF of Δy · x=2: -0.316 PACF of Δy · x=3: 0.188 PACF of Δy · x=4: -0.04 PACF of Δy · x=5: -0.048 PACF of Δy · x=6: -0.044 PACF of Δy · x=7: -0.069 PACF of Δy · x=8: 0.137 PACF of Δy · x=9: 0.065 PACF of Δy · x=10: -0.034 PACF of Δy · x=11: 0.083 PACF of Δy · x=12: 0.019 PACF of Δy · x=13: 0.085 PACF of Δy · x=14: -0.001 PACF of Δy · x=15: 0.052 PACF of Δy · x=16: 0.034 PACF of Δy · x=17: 0.004 PACF of Δy · x=18: 0.019 PACF of Δy · x=19: 0.046 PACF of Δy · x=20: -0.058 PACF of Δy · x=21: 0.023 PACF of Δy · x=22: 0.044 PACF of Δy · x=23: -0.008 PACF of Δy · x=24: -0.022 PACF of Δy · x=25: 0.05 PACF of Δy · x=26: -0.06 PACF of Δy · x=27: 0.065 PACF of Δy · x=28: -0.034 PACF of Δy · x=29: 0.002 PACF of Δy · x=30: -0.004 +band · x=0.5: 0.098 +band · x=30.5: 0.098 -band · x=0.5: -0.098 -band · x=30.5: -0.098 -band +band PACF of Δy

## MA = lagged innovations, not a moving average
In the model $w_t = c+\phi w_{t-1}+e_t+\theta e_{t-1}$ the MA term is yesterday's shock, not a moving average of the data. This is visible from the Wold representation: the weights $\psi_j$ of the impulse response follow $\psi_0=1$, $\psi_j=\theta_j+\sum_i\phi_i\psi_{j-i}$. For ARMA(1,1): $\psi_1=\phi+\theta$, then geometric decay with ratio $\phi$.

In [ ]:
double[] Psi(double[] phi, double[] theta, int n)
{
    var psi = new double[n];
    psi[0] = 1;
    for (var j = 1; j < n; j++)
    {
        psi[j] = j <= theta.Length ? theta[j - 1] : 0;
        for (var i = 1; i <= phi.Length && i <= j; i++) psi[j] += phi[i - 1] * psi[j - i];
    }
    return psi;
}
Chart.Stems("ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15)", Psi(new[] { 0.5 }, new[] { 0.4 }, 16), firstLag: 0);

ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) 0 0.2 0.4 0.6 0.8 1 0 5 10 15 Lag ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=0: 1 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=1: 0.9 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=2: 0.45 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=3: 0.225 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=4: 0.113 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=5: 0.056 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=6: 0.028 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=7: 0.014 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=8: 0.007 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=9: 0.004 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=10: 0.002 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=11: 8.8e-4 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=12: 4.4e-4 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=13: 2.2e-4 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=14: 1.1e-4 ψ_j weights of ARMA(1,1), φ=0.5, θ=0.4 (lag 0…15) · x=15: 5.5e-5

## Estimation and order selection
`FitHannanRissanen` is linear and involves no optimisation:
1. it estimates a long AR with OLS to reconstruct the innovations $\hat e_t$;
2. it estimates by OLS $w_t$ on the lagged $w$ and the lagged $\hat e$.

The regressions use **Householder QR**: the normal equations $X^\top X$ would square the condition number. To compare orders an approximate AIC $n\ln\hat\sigma^2+2k$ and the Ljung-Box test on the residuals are used.

In [ ]:
var candidates = new[] { (1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 0), (2, 1, 2) };
Table.Show(new[] { "ARIMA", "c", "φ", "θ", "σ²", "AIC≈", "Ljung-Box p (20)" }, candidates.Select(o =>
{
    var (p, d, q) = o;
    var m = ArimaModel.FitHannanRissanen(y, p, d, q);
    var res = m.Residuals(y)[Math.Max(p, q)..];
    var aic = res.Length * Math.Log(m.InnovationVariance) + 2 * (p + q + 1);
    return new object[] { $"({p},{d},{q})", m.Constant, string.Join(" ", m.Phi.Select(v => v.ToString("F3"))), string.Join(" ", m.Theta.Select(v => v.ToString("F3"))), m.InnovationVariance, aic, SeriesStatistics.LjungBox(res, 20, p + q).PValue };
}));

ARIMA,c,φ,θ,σ²,AIC≈,Ljung-Box p (20)
"(1,1,0)",0.1664,0.668,,1.1076,44.7673,0
"(0,1,1)",0.5853,,0.940,1.6223,197.0583,0
"(1,1,1)",0.3545,0.398,0.542,0.9877,1.0657,0.0342
"(2,1,0)",0.2206,0.874 -0.314,,0.9949,3.9668,0.0011
"(2,1,2)",0.2567,0.719 -0.158,0.222 -0.165,0.9748,-0.1543,0.0147


Reading the table: with $n=400$ Hannan-Rissanen is consistent but noisy on the individual parameters, whereas $\psi_1=\phi+\theta$ (the first weight of the impulse response) is determined much better. Richer models lower $\hat\sigma^2$ only slightly: the AIC penalises parameters, Ljung-Box checks that no structure is left. Exact maximum likelihood (not implemented here) would start from precisely these estimates.

In [ ]:
var best = ArimaModel.FitHannanRissanen(y[..380], 1, 1, 1);
var fc = best.Forecast(y[..380], 21);
Chart.Lines("ARIMA(1,1,1): 21-step forecast (x axis: t − 340)", new Line("observed", y[340..]), new Line("forecast", fc, Start: 40, Dashed: true));
Console.WriteLine($"out-of-sample RMSE: ARIMA {Accuracy.Rmse(y[380..], fc):F3}; naive {Accuracy.Rmse(y[380..], ReferenceForecasts.Naive(y[..380], 21)):F3}; drift {Accuracy.Rmse(y[380..], ReferenceForecasts.Drift(y[..380], 21)):F3}");

ARIMA(1,1,1): 21-step forecast (x axis: t − 340) 220 240 260 280 300 0 20 40 60 <path d='M64 185.77L74.53 183.79L85.07 184.56L95.6 186.85L106.13 191.03L116.67 194.26L127.2 195.16L137.73 193.59L148.27 191.67L158.8 188.04L169.33 182.65L179.87 179.32L190.4 175.94L200.93 175.04L211.47 174.38L222 172.98L232.53 172.03L243.07 169.46L253.6 164.68L264.13 160.65L274.67 157L285.2 151.41L295.73 147.21L306.27 144.36L316.8 141.6L327.33 140.87L337.87 143.6L348.4 140.77L358.93 137.21L369.47 133.45L380 129.92L390.53 125.79L401.07 121.91L411.6 120.59L422.13 121.85L432.67 121.59L443.2 121.3L453.73 120.59L464.27 118.2L474.8 114.97L485.33 112.75L495.87 111.69L506.4 111.61L516.93 112.32L527.47 114.51L538 115.96L548.53 114.24L559.07 112.93L569.6 111.05L580.13 107.12L590.67 102.01L601.2 96.72L611.73 93.1L622.27 92.45L632.8 91.1L643.33 86.5L653.87 84.65L664.4 86.45L674.93 87.33L685.47 83.82L696 79.85' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> observed · x=0: 233 observed · x=1: 235 observed · x=2: 234 observed · x=3: 233 observed · x=4: 231 observed · x=5: 229 observed · x=6: 228 observed · x=7: 229 observed · x=8: 230 observed · x=9: 232 observed · x=10: 235 observed · x=11: 237 observed · x=12: 239 observed · x=13: 239 observed · x=14: 240 observed · x=15: 241 observed · x=16: 241 observed · x=17: 243 observed · x=18: 245 observed · x=19: 247 observed · x=20: 249 observed · x=21: 253 observed · x=22: 255 observed · x=23: 256 observed · x=24: 258 observed · x=25: 258 observed · x=26: 257 observed · x=27: 258 observed · x=28: 260 observed · x=29: 263 observed · x=30: 264 observed · x=31: 267 observed · x=32: 269 observed · x=33: 270 observed · x=34: 269 observed · x=35: 269 observed · x=36: 269 observed · x=37: 270 observed · x=38: 271 observed · x=39: 273 observed · x=40: 274 observed · x=41: 275 observed · x=42: 275 observed · x=43: 274 observed · x=44: 273 observed · x=45: 272 observed · x=46: 273 observed · x=47: 274 observed · x=48: 275 observed · x=49: 277 observed · x=50: 280 observed · x=51: 283 observed · x=52: 285 observed · x=53: 285 observed · x=54: 286 observed · x=55: 289 observed · x=56: 290 observed · x=57: 289 observed · x=58: 288 observed · x=59: 290 observed · x=60: 292 forecast · x=40: 274 forecast · x=41: 275 forecast · x=42: 276 forecast · x=43: 276 forecast · x=44: 277 forecast · x=45: 278 forecast · x=46: 278 forecast · x=47: 279 forecast · x=48: 279 forecast · x=49: 280 forecast · x=50: 280 forecast · x=51: 281 forecast · x=52: 282 forecast · x=53: 282 forecast · x=54: 283 forecast · x=55: 283 forecast · x=56: 284 forecast · x=57: 284 forecast · x=58: 285 forecast · x=59: 286 forecast · x=60: 286 forecast observed

out-of-sample RMSE: ARIMA 3.808; naive 10.625; drift 5.357

## VAR: several series, cross lags
$$Y_t = c + A_1 Y_{t-1} + \dots + A_p Y_{t-p} + \varepsilon_t.$$
With the same regressors in every equation, equation-by-equation OLS coincides with GLS/SUR (Zellner). A VAR(1) is stationary if the eigenvalues of $A_1$ lie inside the unit circle; the stationary mean is $\mu=(I-A_1)^{-1}c$. The impulse response $A_1^h e_j$ shows how a shock to one series propagates to the other.

In [ ]:
var a = new double[,] { { 0.5, 0.3 }, { -0.2, 0.6 } };
var trueVar = new VarModel<double>(new[] { 1.0, 0.5 }, new[] { a }, new double[2, 2]);
var data = Simulation.Var(new Random(1980), trueVar, 500, 1);
var fit = VarModel.Fit(data, 1);
Table.Show(new[] { "", "A[0,0]", "A[0,1]", "A[1,0]", "A[1,1]", "c0", "c1" }, new[]
{
    new object[] { "true", 0.5, 0.3, -0.2, 0.6, 1.0, 0.5 },
    new object[] { "OLS", fit.Coefficients[0][0, 0], fit.Coefficients[0][0, 1], fit.Coefficients[0][1, 0], fit.Coefficients[0][1, 1], fit.Intercept[0], fit.Intercept[1] },
});
Console.WriteLine($"Stable: {fit.IsStableOrder1Bivariate()}  (true eigenvalues 0.55 ± 0.24i, modulus 0.60)");
var x = Enumerable.Range(0, 120).Select(t => data[t, 0]).ToArray();
var z = Enumerable.Range(0, 120).Select(t => data[t, 1]).ToArray();
Chart.Lines("Two series of a VAR(1) (first 120 steps)", new Line("x", x), new Line("y", z));
// Response to a unit impulse on x at time 0.
var irf = fit.Forecast(new double[,] { { 1 - 0, 0 } }, 15);
var mu = fit.Forecast(new double[,] { { 0, 0 } }, 15);
Chart.Lines("Response to a unit shock on x", new Line("x", Enumerable.Range(0, 15).Select(h => irf[h, 0] - mu[h, 0]).ToArray()), new Line("y", Enumerable.Range(0, 15).Select(h => irf[h, 1] - mu[h, 1]).ToArray()));

,"A[0,0]","A[0,1]","A[1,0]","A[1,1]",c0,c1
true,0.5,0.3,-0.2,0.6,1,0.5
OLS,0.5363,0.3022,-0.221,0.5976,0.8961,0.5902


Stable: True  (true eigenvalues 0.55 ± 0.24i, modulus 0.60)

Two series of a VAR(1) (first 120 steps) -4 -2 0 2 4 6 0 25 50 75 100 125 <path d='M64 108.71L69.06 119.89L74.11 121.65L79.17 141.15L84.22 183.08L89.28 153.13L94.34 133.74L99.39 137.8L104.45 121.2L109.5 130.62L114.56 147.65L119.62 119.78L124.67 155.83L129.73 144.3L134.78 139.69L139.84 121.76L144.9 126.67L149.95 124.05L155.01 123.51L160.06 117.83L165.12 102.33L170.18 90.72L175.23 110.02L180.29 137.56L185.34 90.35L190.4 77.26L195.46 103.27L200.51 148.8L205.57 151.97L210.62 146.99L215.68 146.06L220.74 143L225.79 137.24L230.85 100.59L235.9 138.64L240.96 120.67L246.02 106.96L251.07 133.17L256.13 106.56L261.18 122.89L266.24 121.15L271.3 126.45L276.35 143.36L281.41 154.86L286.46 146.59L291.52 120.35L296.58 127.92L301.63 118.19L306.69 133.92L311.74 166.69L316.8 142.5L321.86 151.98L326.91 151.39L331.97 131.68L337.02 109.95L342.08 123.71L347.14 103.72L352.19 102.78L357.25 93.93L362.3 108.2L367.36 121.23L372.42 144.23L377.47 146.82L382.53 126.1L387.58 126.28L392.64 126.24L397.7 99.77L402.75 106.88L407.81 80.47L412.86 66.8L417.92 114.15L422.98 98.73L428.03 92.96L433.09 99.12L438.14 119.35L443.2 135.48L448.26 139.37L453.31 129.92L458.37 144.64L463.42 140.22L468.48 162.37L473.54 150.16L478.59 131.39L483.65 126.77L488.7 145.44L493.76 142.21L498.82 140.96L503.87 125.35L508.93 121.56L513.98 157.94L519.04 160.09L524.1 135.45L529.15 111.97L534.21 124.26L539.26 104.33L544.32 108.25L549.38 78.15L554.43 81.59L559.49 112.53L564.54 106.94L569.6 119.76L574.66 113.45L579.71 137.25L584.77 115.01L589.82 148.45L594.88 117.62L599.94 123.58L604.99 122.35L610.05 114.2L615.1 114.1L620.16 109.34L625.22 142.58L630.27 129.33L635.33 138.32L640.38 141.11L645.44 154.61L650.5 145.84L655.55 112.61L660.61 104.47L665.66 124.3' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 146.04L69.06 127.67L74.11 143.84L79.17 168.38L84.22 188L89.28 151.27L94.34 135.65L99.39 149.57L104.45 159.04L109.5 167.89L114.56 184.44L119.62 156.11L124.67 156.73L129.73 147.35L134.78 120.41L139.84 134.74L144.9 135.92L149.95 149.39L155.01 126.89L160.06 129.14L165.12 130.04L170.18 148.97L175.23 158.72L180.29 140.68L185.34 138.85L190.4 154.44L195.46 147.66L200.51 172.7L205.57 140.39L210.62 142.52L215.68 151.27L220.74 145.22L225.79 106.32L230.85 125.94L235.9 122.16L240.96 109.99L246.02 130.41L251.07 149.9L256.13 156.77L261.18 167.43L266.24 161.66L271.3 180.25L276.35 159.37L281.41 135.47L286.46 129.23L291.52 141.33L296.58 151.26L301.63 150.9L306.69 171.39L311.74 163.75L316.8 197L321.86 192.01L326.91 170.99L331.97 154.21L337.02 129.23L342.08 142.74L347.14 134.01L352.19 151.57L357.25 158.11L362.3 152.42L367.36 165.77L372.42 135.37L377.47 120.57L382.53 131.82L387.58 152.17L392.64 131.36L397.7 134.39L402.75 113.96L407.81 105.32L412.86 126.13L417.92 144.33L422.98 142.87L428.03 165.66L433.09 155.85L438.14 180.7L443.2 150.87L448.26 159.99L453.31 129.74L458.37 140.92L463.42 153.82L468.48 140.37L473.54 131.46L478.59 156.76L483.65 150.52L488.7 137.43L493.76 142.33L498.82 147.31L503.87 142.72L508.93 159.35L513.98 152.02L519.04 133.15L524.1 119.54L529.15 92.17L534.21 120.34L539.26 136.07L544.32 130.53L549.38 137.1L554.43 150.82L559.49 152.08L564.54 133.54L569.6 150.2L574.66 180.64L579.71 153.23L584.77 142.19L589.82 137.62L594.88 148.32L599.94 158.19L604.99 161.8L610.05 159.83L615.1 135.05L620.16 127.65L625.22 124.29L630.27 121.95L635.33 145.65L640.38 159.03L645.44 158.61L650.5 155.44L655.55 147.19L660.61 181.77L665.66 184.28' fill='none' stroke='#eb6834' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> x · x=0: 3.034 x · x=1: 2.258 x · x=2: 2.136 x · x=3: 0.781 x · x=4: -2.13 x · x=5: -0.051 x · x=6: 1.296 x · x=7: 1.014 x · x=8: 2.167 x · x=9: 1.512 x · x=10: 0.33 x · x=11: 2.265 x · x=12: -0.238 x · x=13: 0.563 x · x=14: 0.883 x · x=15: 2.128 x · x=16: 1.787 x · x=17: 1.968 x · x=18: 2.007 x · x=19: 2.401 x · x=20: 3.477 x · x=21: 4.283 x · x=22: 2.943 x · x=23: 1.031 x · x=24: 4.309 x · x=25: 5.218 x · x=26: 3.412 x · x=27: 0.25 x · x=2

Response to a unit shock on x -0.4 -0.2 0 0.2 0.4 0.6 0 5 10 15 x · x=0: 0.536 x · x=1: 0.221 x · x=2: 0.043 x · x=3: -0.037 x · x=4: -0.059 x · x=5: -0.052 x · x=6: -0.036 x · x=7: -0.021 x · x=8: -0.01 x · x=9: -0.003 x · x=10: 4.5e-4 x · x=11: 0.002 x · x=12: 0.002 x · x=13: 0.001 x · x=14: 8.0e-4 y · x=0: -0.221 y · x=1: -0.251 y · x=2: -0.199 y · x=3: -0.128 y · x=4: -0.068 y · x=5: -0.028 y · x=6: -0.005 y · x=7: 0.005 y · x=8: 0.008 y · x=9: 0.007 y · x=10: 0.005 y · x=11: 0.003 y · x=12: 0.001 y · x=13: 3.7e-4 y · x=14: -6.4e-5 y x

## ARCH/GARCH: volatility, not mean
$$\varepsilon_t=\sigma_t z_t,\qquad \sigma_t^2=\omega+\alpha\varepsilon_{t-1}^2+\beta\sigma_{t-1}^2 .$$
Returns are almost uncorrelated, but their **squares** are not: large shocks follow large shocks (volatility clustering). The variance forecast decays geometrically, with ratio $\alpha+\beta$, towards $\bar\sigma^2=\omega/(1-\alpha-\beta)$.

Estimation uses **variance targeting**: $\omega=s^2(1-\alpha-\beta)$ is fixed, so the maximum likelihood reduces to two parameters, searched on a refined grid.

In [ ]:
var garchTrue = new Garch11<double>(0.05, 0.1, 0.85);
var eps = Simulation.Garch(new Random(1986), garchTrue, 3000);
var g = Garch11.FitVarianceTargeting(eps);
var sigma = g.ConditionalVariances(eps).Select(Math.Sqrt).ToArray();
Chart.Panels("Simulated returns and estimated conditional volatility", new Line("ε_t", eps[..600]), new Line("σ_t estimated", sigma[..600]));
var b = 1.96 / Math.Sqrt(eps.Length);
Chart.Stems("ACF of ε: almost zero", SeriesStatistics.Acf(eps, 20)[1..], b);
Chart.Stems("ACF of ε²: volatility clustering", SeriesStatistics.Acf(eps.Select(e => e * e).ToArray(), 20)[1..], b);
Table.Show(new[] { "", "ω", "α", "β", "α+β", "σ̄²", "log-lik" }, new[]
{
    new object[] { "true", garchTrue.Omega, garchTrue.Alpha, garchTrue.Beta, garchTrue.Persistence, garchTrue.UnconditionalVariance, garchTrue.LogLikelihood(eps) },
    new object[] { "estimated", g.Omega, g.Alpha, g.Beta, g.Persistence, g.UnconditionalVariance, g.LogLikelihood(eps) },
});
Chart.Lines("Forecast of σ² towards the unconditional variance", new Line("σ²_{T+h}", g.ForecastVariance(eps, 100)), new Line("σ̄²", Enumerable.Repeat(g.UnconditionalVariance, 100).ToArray(), Dashed: true));

Simulated returns and estimated conditional volatility ε_t -4 -2 0 2 4 0 200 400 600 <path d='M64 78.5L65.05 100.33L66.11 88.6L67.16 102.91L68.21 75.63L69.27 89.77L70.32 82.48L71.37 98.19L72.43 71.97L73.48 90.87L74.53 73.06L75.59 89.91L76.64 94.91L77.69 66.2L78.75 78.92L79.8 96.85L80.85 77.68L81.91 119.38L82.96 67.98L84.01 85.65L85.07 56.95L86.12 96.11L87.17 84.56L88.23 88.95L89.28 77.95L90.33 92.65L91.39 86.43L92.44 100.94L93.49 90.79L94.55 79.24L95.6 79.93L96.65 98.86L97.71 86.58L98.76 103.9L99.81 89L100.87 72.44L101.92 84.68L102.97 80.37L104.03 85.69L105.08 90.95L106.13 92.9L107.19 78.41L108.24 112.05L109.29 91.35L110.35 89.7L111.4 99.31L112.45 101.2L113.51 65.67L114.56 75.07L115.61 91.78L116.67 90.58L117.72 82.72L118.77 84.84L119.83 92.92L120.88 76.31L121.93 78.47L122.99 82.67L124.04 98.46L125.09 97.88L126.15 82.1L127.2 81.54L128.25 92.91L129.31 97.01L130.36 81.25L131.41 105.43L132.47 89.49L133.52 75.42L134.57 82.98L135.63 99.71L136.68 84.18L137.73 75.11L138.79 84.25L139.84 95.94L140.89 96.24L141.95 96.59L143 77.31L144.05 99.31L145.11 78L146.16 96.12L147.21 81.48L148.27 100.2L149.32 79.13L150.37 88.7L151.43 99.27L152.48 98.15L153.53 85.86L154.59 98.02L155.64 92.82L156.69 90.15L157.75 88.03L158.8 86.29L159.85 93.76L160.91 87.58L161.96 90.42L163.01 92.64L164.07 80.98L165.12 92.16L166.17 85.58L167.23 81.69L168.28 82.86L169.33 79.1L170.39 90.44L171.44 87.55L172.49 87.45L173.55 83.32L174.6 76.49L175.65 83.48L176.71 104.55L177.76 84.28L178.81 92.32L179.87 96.12L180.92 92.4L181.97 89.09L183.03 85.05L184.08 80.22L185.13 85.89L186.19 88.15L187.24 81.19L188.29 82.1L189.35 83.11L190.4 119.7L191.45 64.47L192.51 94.86L193.56 81.4L194.61 106.22L195.67 95.61L196.72 87.72L197.77 83.52L198.83 96.69L199.88 82.66L200.93 82.59L201.99 93.59L203.04 87.48L204.09 62.66L205.15 85.86L206.2 96.28L207.25 77.67L208.31 100.26L209.36 71.98L210.41 100.18L211.47 102.84L212.52 78.74L213.57 96.74L214.63 64.14L215.68 91.55L216.73 79.24L217.79 107.5L218.84 60.46L219.89 97.83L220.95 91.55L222 95.86L223.05 68.42L224.11 89.98L225.16 78.9L226.21 98.25L227.27 90.09L228.32 78.32L229.37 88.59L230.43 100.76L231.48 90.4L232.53 61.27L233.59 88.87L234.64 79.4L235.69 72.72L236.75 92.25L237.8 87.71L238.85 87.47L239.91 78.17L240.96 104.63L242.01 76.97L243.07 84.79L244.12 99.12L245.17 100.43L246.23 84.78L247.28 86.02L248.33 87.14L249.39 99.02L250.44 94.5L251.49 71.85L252.55 90.58L253.6 97.31L254.65 63.71L255.71 79.61L256.76 98.15L257.81 97.41L258.87 85.66L259.92 66.83L260.97 76.09L262.03 103.6L263.08 82.6L264.13 74.28L265.19 77.2L266.24 94.67L267.29 101.67L268.35 76.66L269.4 90.67L270.45 88.84L271.51 90.75L272.56 92.02L273.61 91.45L274.67 86.99L275.72 87.77L276.77 88.63L277.83 102.82L278.88 104.76L279.93 91.66L280.99 103.46L282.04 104.28L283.09 78.62L284.15 88.25L285.2 82.55L286.25 98.66L287.31 96.79L288.36 91.05L289.41 86.49L290.47 92.79L291.52 96.06L292.57 101.57L293.63 80.24L294.68 78.33L295.73 78.03L296.79 76.65L297.84 103.46L298.89 107.53L299.95 75.22L301 88.79L302.05 88.81L303.11 83.56L304.16 85.37L305.21 92.02L306.27 87.56L307.32 84.36L308.37 92.79L309.43 89.26L310.48 96.91L311.53 88.98L312.59 83.94L313.64 85.16L314.69 82.43L315.75 88.91L316.8 92.13L317.85 92.13L318.91 99.58L319.96 85.7L321.01 87.06L322.07 88.35L323.12 80.68L324.17 100.27L325.23 101.29L326.28 86.77L327.33 96.63L328.39 78.54L329.44 94.66L330.49 83.33L331.55 89.67L332.6 97.73L333.65 98.01L334.71 93.66L335.76 88.11L336.81 81.61L337.87 79.01L338.92 93.34L339.97 83.28L341.03 91.33L342.08 87.03L343.13 90.1L344.19 87.85L345.24 84.45L346.29 74.76L347.35 97.82L348.4 86.43L349.45 88.25L350.51 100.79L351.56 80.78L352.61 96.86L353.67 81.72L354.72 95.78L355.77 100.74L356.83 96.76L357.88 75.98L358.93 85.22L359.99 87.5L361.04 88.53L362.09 113.39L363.15 89.47L364.2 75.58L365.25 80.67L366.31 108.13L367.36 90.13L368.41 71.57L369.47 92.78L370.52 78.06L371.57 81.67L372.63 78.12L373.68 86.71L374.73 100.04L375.79 97.04L376.84 79.51L377.89 86.67L378.95 76.52L380 77.25L381.05 97.07L382.11 101.49L383.16 100.11L384.21 75.59L3

ACF of ε: almost zero -0.06 -0.04 -0.02 0 0.02 0.04 5 10 15 20 Lag ACF of ε: almost zero · x=1: 0.01 ACF of ε: almost zero · x=2: -0.002 ACF of ε: almost zero · x=3: 0.002 ACF of ε: almost zero · x=4: 0.019 ACF of ε: almost zero · x=5: 0.015 ACF of ε: almost zero · x=6: 0.027 ACF of ε: almost zero · x=7: -0.003 ACF of ε: almost zero · x=8: 0.012 ACF of ε: almost zero · x=9: 0.022 ACF of ε: almost zero · x=10: -0.022 ACF of ε: almost zero · x=11: -0.047 ACF of ε: almost zero · x=12: -0.004 ACF of ε: almost zero · x=13: 0.024 ACF of ε: almost zero · x=14: -1.3e-4 ACF of ε: almost zero · x=15: 0.022 ACF of ε: almost zero · x=16: 0.001 ACF of ε: almost zero · x=17: -0.043 ACF of ε: almost zero · x=18: -0.04 ACF of ε: almost zero · x=19: -0.004 ACF of ε: almost zero · x=20: -0.05 +band · x=0.5: 0.036 +band · x=20.5: 0.036 -band · x=0.5: -0.036 -band · x=20.5: -0.036 -band +band ACF of ε: almost zero

ACF of ε²: volatility clustering -0.05 0 0.05 0.1 0.15 0.2 5 10 15 20 Lag ACF of ε²: volatility clustering · x=1: 0.17 ACF of ε²: volatility clustering · x=2: 0.133 ACF of ε²: volatility clustering · x=3: 0.169 ACF of ε²: volatility clustering · x=4: 0.174 ACF of ε²: volatility clustering · x=5: 0.085 ACF of ε²: volatility clustering · x=6: 0.109 ACF of ε²: volatility clustering · x=7: 0.105 ACF of ε²: volatility clustering · x=8: 0.067 ACF of ε²: volatility clustering · x=9: 0.081 ACF of ε²: volatility clustering · x=10: 0.119 ACF of ε²: volatility clustering · x=11: 0.084 ACF of ε²: volatility clustering · x=12: 0.097 ACF of ε²: volatility clustering · x=13: 0.07 ACF of ε²: volatility clustering · x=14: 0.086 ACF of ε²: volatility clustering · x=15: 0.095 ACF of ε²: volatility clustering · x=16: 0.072 ACF of ε²: volatility clustering · x=17: 0.064 ACF of ε²: volatility clustering · x=18: 0.067 ACF of ε²: volatility clustering · x=19: 0.058 ACF of ε²: volatility clustering · x=20: 0.038 +band · x=0.5: 0.036 +band · x=20.5: 0.036 -band · x=0.5: -0.036 -band · x=20.5: -0.036 -band +band ACF of ε²: volatility clustering

,ω,α,β,α+β,σ̄²,log-lik
true,0.05,0.1,0.85,0.95,1,-4023.3777
estimated,0.0524,0.0921,0.8506,0.9427,0.9144,-4022.5675


Forecast of σ² towards the unconditional variance 0.8 0.825 0.85 0.875 0.9 0.925 0 20 40 60 80 100 <path d='M64 208.77L70.32 201.3L76.64 194.25L82.96 187.6L89.28 181.34L95.6 175.43L101.92 169.86L108.24 164.61L114.56 159.67L120.88 155L127.2 150.6L133.52 146.46L139.84 142.55L146.16 138.87L152.48 135.4L158.8 132.12L165.12 129.04L171.44 126.13L177.76 123.38L184.08 120.8L190.4 118.36L196.72 116.06L203.04 113.9L209.36 111.86L215.68 109.93L222 108.12L228.32 106.41L234.64 104.79L240.96 103.27L247.28 101.84L253.6 100.49L259.92 99.22L266.24 98.02L272.56 96.88L278.88 95.82L285.2 94.81L291.52 93.86L297.84 92.97L304.16 92.13L310.48 91.33L316.8 90.58L323.12 89.88L329.44 89.21L335.76 88.59L342.08 87.99L348.4 87.44L354.72 86.91L361.04 86.42L367.36 85.95L373.68 85.51L380 85.09L386.32 84.7L392.64 84.33L398.96 83.99L405.28 83.66L411.6 83.35L417.92 83.06L424.24 82.78L430.56 82.52L436.88 82.28L443.2 82.05L449.52 81.83L455.84 81.63L462.16 81.44L468.48 81.25L474.8 81.08L481.12 80.92L487.44 80.77L493.76 80.63L500.08 80.49L506.4 80.36L512.72 80.24L519.04 80.13L525.36 80.02L531.68 79.92L538 79.83L544.32 79.74L550.64 79.65L556.96 79.57L563.28 79.5L569.6 79.43L575.92 79.36L582.24 79.3L588.56 79.24L594.88 79.18L601.2 79.13L607.52 79.08L613.84 79.03L620.16 78.99L626.48 78.95L632.8 78.91L639.12 78.87L645.44 78.84L651.76 78.8L658.08 78.77L664.4 78.74L670.72 78.72L677.04 78.69L683.36 78.67L689.68 78.64' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 78.26L70.32 78.26L76.64 78.26L82.96 78.26L89.28 78.26L95.6 78.26L101.92 78.26L108.24 78.26L114.56 78.26L120.88 78.26L127.2 78.26L133.52 78.26L139.84 78.26L146.16 78.26L152.48 78.26L158.8 78.26L165.12 78.26L171.44 78.26L177.76 78.26L184.08 78.26L190.4 78.26L196.72 78.26L203.04 78.26L209.36 78.26L215.68 78.26L222 78.26L228.32 78.26L234.64 78.26L240.96 78.26L247.28 78.26L253.6 78.26L259.92 78.26L266.24 78.26L272.56 78.26L278.88 78.26L285.2 78.26L291.52 78.26L297.84 78.26L304.16 78.26L310.48 78.26L316.8 78.26L323.12 78.26L329.44 78.26L335.76 78.26L342.08 78.26L348.4 78.26L354.72 78.26L361.04 78.26L367.36 78.26L373.68 78.26L380 78.26L386.32 78.26L392.64 78.26L398.96 78.26L405.28 78.26L411.6 78.26L417.92 78.26L424.24 78.26L430.56 78.26L436.88 78.26L443.2 78.26L449.52 78.26L455.84 78.26L462.16 78.26L468.48 78.26L474.8 78.26L481.12 78.26L487.44 78.26L493.76 78.26L500.08 78.26L506.4 78.26L512.72 78.26L519.04 78.26L525.36 78.26L531.68 78.26L538 78.26L544.32 78.26L550.64 78.26L556.96 78.26L563.28 78.26L569.6 78.26L575.92 78.26L582.24 78.26L588.56 78.26L594.88 78.26L601.2 78.26L607.52 78.26L613.84 78.26L620.16 78.26L626.48 78.26L632.8 78.26L639.12 78.26L645.44 78.26L651.76 78.26L658.08 78.26L664.4 78.26L670.72 78.26L677.04 78.26L683.36 78.26L689.68 78.26' fill='none' stroke='#eb6834' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/> σ²_{T+h} · x=0: 0.801 σ²_{T+h} · x=1: 0.808 σ²_{T+h} · x=2: 0.814 σ²_{T+h} · x=3: 0.819 σ²_{T+h} · x=4: 0.825 σ²_{T+h} · x=5: 0.83 σ²_{T+h} · x=6: 0.835 σ²_{T+h} · x=7: 0.839 σ²_{T+h} · x=8: 0.844 σ²_{T+h} · x=9: 0.848 σ²_{T+h} · x=10: 0.852 σ²_{T+h} · x=11: 0.855 σ²_{T+h} · x=12: 0.859 σ²_{T+h} · x=13: 0.862 σ²_{T+h} · x=14: 0.865 σ²_{T+h} · x=15: 0.868 σ²_{T+h} · x=16: 0.87 σ²_{T+h} · x=17: 0.873 σ²_{T+h} · x=18: 0.875 σ²_{T+h} · x=19: 0.877 σ²_{T+h} · x=20: 0.88 σ²_{T+h} · x=21: 0.882 σ²_{T+h} · x=22: 0.883 σ²_{T+h} · x=23: 0.885 σ²_{T+h} · x=24: 0.887 σ²_{T+h} · x=25: 0.888 σ²_{T+h} · x=26: 0.89 σ²_{T+h} · x=27: 0.891 σ²_{T+h} · x=28: 0.893 σ²_{T+h} · x=29: 0.894 σ²_{T+h} · x=30: 0.895 σ²_{T+h} · x=31: 0.896 σ²_{T+h} · x=32: 0.897 σ²_{T+h} · x=33: 0.898 σ²_{T+h} · x=34: 0.899 σ²_{T+h} · x=35: 0.9 σ²_{T+h} · x=36: 0.901 σ²_{T+h} · x=37: 0.902 σ²_{T+h} · x=38: 0.902 σ²_{T+h} · x=39: 0.903 σ²_{T+h} · x=40: 0.904 σ²_{T+h} · x=41: 0.904 σ²_{T+h} · x=42: 0.905 σ²_{T+h} · x=43: 0.905 σ²_{T+h} · x=44: 0.906 σ²_{T+h} · x=45: 0.906 σ²_{T+h} · x=46: 0.907 σ²_{T+h} · x=47: 0.907 σ²_{T+h} · x=48: 0.908 σ²_{T+